### Lab 9.1 Attention Implementation

This week you will experiment with attention-based models.

In [1]:
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F
import math

1. Complete the following implementation of scaled dot-product attention.   Run the code cell to verify that the output shape is what it should be.

*Note: you can use `scores = scores.masked_fill(...)` to fill in values where the mask is True.  Fill in -1e9 as the score for masked values.*

In [2]:
def attention(Q,K,V,mask=None):
    """
    Computes scaled dot-product attention.

    Compute scores as Q*K^T.
    Optionally mask out score values to -1e9 where the mask is True.
    Divide by sqrt(d_k).
    Compute softmax on scores along the rows to obtain attention weights.
    Matrix multiply attention weights by values.

    Arguments:
        Q: queries [B,L,d_k]
        K: keys    [B,S,d_k]
        V: values  [B,S,d_v]
        mask: optional Boolean mask where True means hidden [B,L,S]

    Returns:
        Sequence of context vectors of shape [B,L,d_v]
    """
    d_k = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1))
    scores = scores / np.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)

    return output



Q = torch.rand(1,5,64)
K = torch.rand(1,10,64)
V = torch.rand(1,10,8)
mask = (torch.rand(1,5,10)>0.5)

y = attention(Q,K,V,mask=mask)

y.shape

torch.Size([1, 5, 8])

The following code creates classes to build a Transformer-style decoder for generating sequences.

In [3]:
class AttentionHead(nn.Module):
    def __init__(self,d_model,d_k):
        super().__init__()
        self.WQ = nn.Linear(d_model,d_k)
        self.WK = nn.Linear(d_model,d_k)
        self.WV = nn.Linear(d_model,d_k)

    def forward(self,Q,K,V,mask=None):
        """ Compute attention head.

            Project the input to queries, keys, and values, and then apply attention.
            Arguments:
                Q: queries [B,L,d_model]
                K: keys    [B,S,d_model]
                V: values  [B,L,d_model]
                mask: optional Boolean mask where True means hidden [B,L,S]
            Output:
                Context vectors [B,L,d_k]
        """
        # apply linear projections to queries, keys, and values followed by masked attention
        return attention(self.WQ(Q),self.WK(K),self.WV(V),mask=mask)

class MultiHeadAttention(nn.Module):
    def __init__(self,d_model=512,num_heads=8):
        super().__init__()
        self.heads = []
        d_k = d_model // num_heads
        self.heads = nn.ModuleList([AttentionHead(d_model,d_k) for head in range(num_heads)])
        self.W = nn.Linear(d_model,d_model)

    def forward(self,Q,K,V,mask=None):
        """ Compute multi-head attention.

            Applies attention num_heads times, concatenates the results, and applies a final linear projection.
            Arguments:
                Q: queries [B,L,d_model]
                K: keys    [B,S,d_model]
                V: values  [B,L,d_model]
                mask: optional Boolean mask where True means hidden [B,L,S]
            Output:
               result of multi-head attention [B,L,d_model]
        """
        # compute each attention head and concatenate
        h = torch.cat([head(Q,K,V,mask=mask) for head in self.heads],dim=-1)

        # apply output projection
        return self.W(h)

class SelfAttentionBlock(nn.Module):
    def __init__(self,d_model=512,num_heads=8,d_ff=2048):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model,num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.ReLU(),
            nn.Linear(d_ff,d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self,x,mask=None):
        """ Compute self attention block.

            Arguments:
                x: input sequence [B,S,d_model]
                mask: optional Boolean mask where True means hidden [B,L,S]
            Output:
               result of attention block [B,L,d_model]
        """
        # compute multi-head attention
        mha = self.multi_head_attention(x,x,x,mask=mask)

        # residual connection and layer normalization
        x = self.ln1(mha + x)

        # compute feed-forward network
        ff = self.feed_forward(x)

        # residual connection and layer normalization
        x = self.ln2(ff + x)

        return x

class PositionalEmbedding(nn.Module):
    def __init__(self,max_seq_len,d_model):
        super().__init__()
        self.positional_embedding = nn.Embedding(max_seq_len,d_model)

    def forward(self,x):
        """ Adds a positional embedding.

            Arguments:
                x: input token sequence [B,S,d_model]
            Output:
               sequence with positional embedding added [B,S,d_model]
        """
        # get sequence length
        N = x.shape[1]

        # look up positional embedding vectors
        pe = self.positional_embedding(torch.arange(N).to(x.device)) # [N,d_model]

        # add to input
        x = x + pe[None,...] # [B,N,d_model]

        return x

class TransformerDecoder(nn.Module):
    def __init__(self,vocabulary_size,max_seq_len,
                      d_model=512,num_heads=8,d_ff=2048,num_blocks=6):
        super().__init__()
        self.blocks = nn.ModuleList([SelfAttentionBlock(d_model,num_heads,d_ff) for b in range(num_blocks)])
        self.token_embedding = nn.Embedding(vocabulary_size,d_model)
        self.output = nn.Linear(d_model,vocabulary_size)
        self.positional_embedding = PositionalEmbedding(max_seq_len,d_model)

    def forward(self,x,mask=None):
        """ Computes the decoded sequence:

            Convert input to token embedding vectors
            Add positional embedding to input
            Apply self-attention blocks with mask
            Compute output

            Arguments:
                x: input token sequence [B,S]
                mask: optional Boolean mask where false means hidden [B,S]
            Output:
               sequence predictions [B,S,output_size]
        """
        # look up embedding vectors for tokens
        x = self.token_embedding(x) # [B,S,d_model]

        # apply positional embedding
        x = self.positional_embedding(x) # [B,S,d_model]

        # apply sequence of masked self-attention blocks
        for block in self.blocks:
            x = block(x,mask=mask) # [B,S,d_model]

        # produce sequence of output vectors
        y = self.output(x) # [B,S,vocabulary_size]

        return y


This function produces masks appropriate for sequence prediction.  The mask ensures that the output token at time t+1 only sees the generated sequence up to time t.

In [4]:
def make_mask(seq_len):
    """ Make a mask for sequence prediction. """
    return (torch.triu(torch.ones((1,seq_len,seq_len)), diagonal=1)==1)

make_mask(10)

tensor([[[False,  True,  True,  True,  True,  True,  True,  True,  True,  True],
         [False, False,  True,  True,  True,  True,  True,  True,  True,  True],
         [False, False, False,  True,  True,  True,  True,  True,  True,  True],
         [False, False, False, False,  True,  True,  True,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True,  True,  True],
         [False, False, False, False, False, False,  True,  True,  True,  True],
         [False, False, False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False, False, False, False,  True,  True],
         [False, False, False, False, False, False, False, False, False,  True],
         [False, False, False, False, False, False, False, False, False, False]]])

Now we will make a sequence of integers and see if the Transformer decoder can learn the sequence.

In [5]:
seq = torch.arange(100)
x = seq[:-1][None,...]
y = seq[1:][None,...]
mask = make_mask(x.shape[1])
x,y

(tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
          18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
          36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
          54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
          72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
          90, 91, 92, 93, 94, 95, 96, 97, 98]]),
 tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
          19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
          37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
          55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72,
          73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90,
          91, 92, 93, 94, 95, 96, 97, 98, 99]]))

In [6]:
steps = 100

model = TransformerDecoder(vocabulary_size=100,max_seq_len=x.shape[1],
                           d_model=64,num_heads=8,d_ff=512,num_blocks=3
                           )


opt = torch.optim.Adam(model.parameters(),lr=.01)
loss_fn = nn.CrossEntropyLoss()

for step in range(steps):
    model.train()
    opt.zero_grad()

    y_pred = model(x,mask)
    loss = loss_fn(y_pred.view(-1,y_pred.shape[-1]),y.view(-1))
    loss.backward()

    opt.step()

    print(step,loss.item())

0 4.766934871673584
1 2.999788522720337
2 2.730412006378174
3 2.4291322231292725
4 1.7258507013320923
5 1.1748794317245483
6 0.8934359550476074
7 0.541538417339325
8 0.3750799894332886
9 0.2570783197879791
10 0.1726658046245575
11 0.11832842975854874
12 0.08654463291168213
13 0.06535380333662033
14 0.04903857409954071
15 0.03652738779783249
16 0.027493080124258995
17 0.02114490605890751
18 0.016653846949338913
19 0.01340195070952177
20 0.010982697829604149
21 0.009145899675786495
22 0.007725749164819717
23 0.0066115581430494785
24 0.005725136958062649
25 0.00501137413084507
26 0.004427576437592506
27 0.003942303359508514
28 0.0035328385420143604
29 0.003183206543326378
30 0.002882078755646944
31 0.0026210083160549402
32 0.002393720904365182
33 0.0021955121774226427
34 0.0020226985216140747
35 0.0018715709447860718
36 0.0017390232533216476
37 0.0016226848820224404
38 0.0015202772337943316
39 0.001429954543709755
40 0.0013501551002264023
41 0.0012794844806194305
42 0.0012168287066742778


If the Transformer has learned the sequence correctly, this output will read 1, 2, 3, ..., 97, 98, 99.

In [7]:
torch.argmax(model(x),-1)

tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
         19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
         37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
         55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72,
         73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90,
         91, 92, 93, 94, 95, 96, 97, 98, 99]])

2. What size context does the Transformer need in order to learn the above sequence?

Needs a context of size 1 to learn the sequence of integers. Each number in the sequence is needed to predict the next one. i.e. previous num + 1

3. Now design a pattern that requires a larger context and see if the Transformer can learn it.

In [10]:
def fibonacci(length=100, start_values=[1,1]):
    seq = torch.zeros(length, dtype=torch.long)
    seq[:len(start_values)] = torch.tensor(start_values)
    for i in range(len(start_values), length):
        seq[i] = (seq[i-1] + seq[i-2]) %100
    return seq



fib = fibonacci()
x_fib = fib[:-1][None,...]
y_fib = fib[1:][None,...]
mask_fib = make_mask(x_fib.shape[1])

x_fib,y_fib

(tensor([[ 1,  1,  2,  3,  5,  8, 13, 21, 34, 55, 89, 44, 33, 77, 10, 87, 97, 84,
          81, 65, 46, 11, 57, 68, 25, 93, 18, 11, 29, 40, 69,  9, 78, 87, 65, 52,
          17, 69, 86, 55, 41, 96, 37, 33, 70,  3, 73, 76, 49, 25, 74, 99, 73, 72,
          45, 17, 62, 79, 41, 20, 61, 81, 42, 23, 65, 88, 53, 41, 94, 35, 29, 64,
          93, 57, 50,  7, 57, 64, 21, 85,  6, 91, 97, 88, 85, 73, 58, 31, 89, 20,
           9, 29, 38, 67,  5, 72, 77, 49, 26]]),
 tensor([[ 1,  2,  3,  5,  8, 13, 21, 34, 55, 89, 44, 33, 77, 10, 87, 97, 84, 81,
          65, 46, 11, 57, 68, 25, 93, 18, 11, 29, 40, 69,  9, 78, 87, 65, 52, 17,
          69, 86, 55, 41, 96, 37, 33, 70,  3, 73, 76, 49, 25, 74, 99, 73, 72, 45,
          17, 62, 79, 41, 20, 61, 81, 42, 23, 65, 88, 53, 41, 94, 35, 29, 64, 93,
          57, 50,  7, 57, 64, 21, 85,  6, 91, 97, 88, 85, 73, 58, 31, 89, 20,  9,
          29, 38, 67,  5, 72, 77, 49, 26, 75]]))

In [11]:
model_fib = TransformerDecoder(vocabulary_size=100, max_seq_len=x_fib.shape[1],
                               d_model=128, num_heads=8, d_ff=512, num_blocks=6)

opt_fib = torch.optim.Adam(model_fib.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_fib, factor=0.5, patience=50)
loss_fn = nn.CrossEntropyLoss()

best_loss = float('inf')
patience = 100
patience_counter = 0
steps = 1000


for step in range(steps):
    model_fib.train()
    opt_fib.zero_grad()

    y_pred = model_fib(x_fib, mask_fib)
    loss = loss_fn(y_pred.view(-1, y_pred.shape[-1]), y_fib.view(-1))
    loss.backward()
    opt_fib.step()

    scheduler.step(loss)

    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.item():.6f}")

    if loss.item() < best_loss:
        best_loss = loss.item()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at step {step}")
        break

model_fib.eval()
with torch.no_grad():
    predicted = torch.argmax(model_fib(x_fib), -1)

print("\nFirst 20 elements of original Fibonacci sequence:")
print(fib[:20])
print("\nFirst 20 elements of predicted sequence:")
print(predicted[0, :20])

correct_elements = (predicted[0] == y_fib[0]).float()
print("\nAccuracy per position (1 = correct, 0 = incorrect):")
print(correct_elements[:20])

overall_accuracy = correct_elements.mean().item() * 100
print(f"\nOverall accuracy: {overall_accuracy:.2f}%")


Step 0, Loss: 4.730315
Step 50, Loss: 0.064097
Step 100, Loss: 0.026899
Step 150, Loss: 0.016309
Step 200, Loss: 0.011124
Step 250, Loss: 0.008138
Step 300, Loss: 0.006243
Step 350, Loss: 0.004957
Step 400, Loss: 0.004040
Step 450, Loss: 0.003361
Step 500, Loss: 0.002844
Step 550, Loss: 0.002439
Step 600, Loss: 0.002116
Step 650, Loss: 0.001854
Step 700, Loss: 0.001638
Step 750, Loss: 0.001458
Step 800, Loss: 0.001306
Step 850, Loss: 0.001176
Step 900, Loss: 0.001065
Step 950, Loss: 0.000969

First 20 elements of original Fibonacci sequence:
tensor([ 1,  1,  2,  3,  5,  8, 13, 21, 34, 55, 89, 44, 33, 77, 10, 87, 97, 84,
        81, 65])

First 20 elements of predicted sequence:
tensor([ 1,  2,  3,  5,  8, 13, 21, 34, 55, 89, 44, 33, 77, 10, 87, 97, 84, 81,
        65, 46])

Accuracy per position (1 = correct, 0 = incorrect):
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.])

Overall accuracy: 100.00%


The transformer learned fibonnaci (addition of previous two wrapped %100 for vocab size)!